# 3.1 Generate validation predictions

**Inputs:** trained component models, selected features and validation features.

**Output:** one Parquet file containing the eight internal component predictions for every horizon.

In [1]:
# Step 1 - Imports and component models

import gc
import os
import sys

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.append(os.path.abspath("../2_Features_build")) ; import target_features

HORIZONS = range(1, variables.HORIZON_COUNT+1)
COMPONENT_MODELS = [
    ("base_l1", "normal", "full_range_regressor_clipped_MAE_loss", "regression"),
    ("base_l2", "arcsinh", "full_range_regressor_unclipped_RMSE_loss", "regression"),
    ("spike_probability", "spikes", "positive_spike_classifier_binary_loss", "probability"),
    ("spike_mae", "spikes", "positive_spike_regressor_unclipped_mae_loss", "regression"),
    ("spike_q90", "spikes", "positive_spike_regressor_unclipped_quantile_loss", "regression"),
    ("dip_probability", "dips", "negative_spike_classifer_unclipped_binary_loss", "probability"),
    ("dip_mae", "dips", "negative_spike_regressor_unclipped_mae_loss", "regression"),
    ("dip_q10", "dips", "negative_spike_regressor_unclipped_quantile_loss", "regression"),
]

In [2]:
# Step 2 - Load validation data

def read_period(path, columns):
    index_col = (pq.ParquetFile(path).schema_arrow.pandas_metadata or {})["index_columns"][0]
    return pd.read_parquet(path, columns=columns, filters=[(index_col, ">", variables.VALID_START), (index_col, "<=", variables.TEST_START)]).sort_index().astype(np.float32)

objectives = list(dict.fromkeys(model[1] for model in COMPONENT_MODELS))
selected_by_objective = {objective: pd.read_parquet(variables.SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet") for objective in objectives}

def selected_features(objective, horizon):
    selected = selected_by_objective[objective]
    return selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()

validation_index = read_period(variables.AGG_TARGET_DATASET_PATH, ["target_h1"]).index

In [3]:
# Step 3 - Predict and save

predictions = {}

for objective in objectives:
    objective_features = sorted({feature for horizon in HORIZONS for feature in selected_features(objective, horizon)})
    features = read_period(variables.FEATURES_DATASET_PATH, objective_features).reindex(validation_index)
    models = [model for model in COMPONENT_MODELS if model[1] == objective]

    for horizon in tqdm(HORIZONS, desc=objective):
        model_input = target_features.append_target_time_feats(features[selected_features(objective, horizon)], horizon)
        for output_name, _, model_name, prediction_type in models:
            model = joblib.load(variables.TRAINED_MODELS_PATH / f"h{horizon:02d}_{model_name}.joblib")
            prediction = model.predict_proba(model_input)[:, 1] if prediction_type == "probability" else np.sinh(model.predict(model_input))*variables.PRICE_TRANSFORM_SCALE
            predictions[f"{output_name}_h{horizon}"] = np.asarray(prediction, dtype=np.float32)
            del model
        del model_input
        gc.collect()

    del features
    gc.collect()

predictions = pd.DataFrame(predictions, index=validation_index)
predictions.index.name = "date"
predictions.to_parquet(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)

display(predictions.head())
print(predictions.shape)
print(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)

dips: 100%|██████████| 96/96 [01:00<00:00,  1.59it/s]


,base_l1_h1,base_l1_h2,base_l1_h3,base_l1_h4,base_l1_h5,base_l1_h6,base_l1_h7,base_l1_h8,base_l1_h9,base_l1_h10,...,dip_q10_h93,dip_probability_h94,dip_mae_h94,dip_q10_h94,dip_probability_h95,dip_mae_h95,dip_q10_h95,dip_probability_h96,dip_mae_h96,dip_q10_h96
date,,,,,,,,,,,,,,,,,,,,,
2024-07-01 00:05:00,64.090195,63.113113,62.592667,64.020103,67.253937,70.928116,72.351891,73.954407,78.195930,84.384293,...,83.571022,0.005175,149.496719,90.521973,0.007308,172.140549,75.785461,0.005682,76.493309,53.774666
2024-07-01 00:10:00,78.125633,68.562943,67.186951,67.265976,68.578712,73.909302,75.061287,74.068298,81.837357,84.457085,...,87.249504,0.005175,154.097000,90.635422,0.007308,169.395386,75.785461,0.005682,76.493309,54.733353
2024-07-01 00:15:00,83.100830,73.343140,68.198517,66.984238,68.462868,75.884430,74.183701,74.418816,81.433868,85.752815,...,83.146408,0.005175,150.463547,90.424225,0.007308,192.918350,75.785461,0.005682,76.493309,54.733353
2024-07-01 00:20:00,87.827690,75.119095,70.791435,67.238319,68.573074,75.950348,75.195938,74.499855,82.239487,85.752815,...,82.496651,0.005175,154.035049,90.543068,0.007308,192.149292,75.785461,0.005682,76.493309,53.774666
2024-07-01 00:25:00,91.239716,75.493912,70.700394,65.193855,68.082062,75.885468,74.603806,74.507912,80.691666,85.821609,...,83.659157,0.005175,161.377762,91.218079,0.007308,192.149292,75.785461,0.005682,73.397354,51.346855


(52992, 768)
/home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/4_combine_models/1_validation_component_predictions.parquet
